# UK Choropleth Maps — Exploration

This notebook produces three Folium choropleth maps and one Plotly Express choropleth for comparison.

**Maps produced:**
- `outputs/maps/imd_map.html` — Index of Multiple Deprivation 2019
- `outputs/maps/income_map.html` — Median household income 2020
- `outputs/maps/unemployment_map.html` — Unemployment claimant rate 2023

**Prerequisite:** Run `python data/download_data.py` first.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import plotly.express as px
from IPython.display import IFrame

from src.choropleth_builder import ChoroplethBuilder

GEOJSON = '../data/geojson/local_authorities_2022.geojson'
OUT = Path('../outputs/maps')
OUT.mkdir(parents=True, exist_ok=True)

## 1. Load and inspect boundary GeoJSON

In [ ]:
gdf = gpd.read_file(GEOJSON)
print(f'Features: {len(gdf):,}  |  CRS: {gdf.crs}')
gdf.head(3)

## 2. Load metric data

In [ ]:
# IMD 2019 — Local Authority Average Score
imd_raw = pd.read_excel('../data/imd_2019.xlsx', sheet_name='IoD2019 Local Authority District')
imd = imd_raw[['Local Authority District code (2019)', 'Average score']].copy()
imd.columns = ['LAD22CD', 'imd_score']
print(f'IMD rows: {len(imd):,}')
imd.head(3)

In [ ]:
# Unemployment claimant count
unemp_raw = pd.read_csv('../data/unemployment_2023.csv')
unemp = unemp_raw[['GEOGRAPHY_CODE', 'OBS_VALUE']].copy()
unemp.columns = ['LAD22CD', 'claimant_pct']
print(f'Unemployment rows: {len(unemp):,}')
unemp.head(3)

## 3. Map 1 — Index of Multiple Deprivation

In [ ]:
builder_imd = ChoroplethBuilder(GEOJSON, imd, join_key='LAD22CD')
m_imd = builder_imd.build(
    column='imd_score',
    legend_name='IMD Average Score (2019)',
    fill_color='YlOrRd',
)
builder_imd.save(OUT / 'imd_map.html')
print('Saved imd_map.html')
IFrame(str(OUT / 'imd_map.html'), width='100%', height=500)

## 4. Map 2 — Unemployment claimant rate

In [ ]:
builder_unemp = ChoroplethBuilder(GEOJSON, unemp, join_key='LAD22CD')
m_unemp = builder_unemp.build(
    column='claimant_pct',
    legend_name='Unemployment Claimant Rate % (Apr 2023)',
    fill_color='RdPu',
)
builder_unemp.save(OUT / 'unemployment_map.html')
print('Saved unemployment_map.html')
IFrame(str(OUT / 'unemployment_map.html'), width='100%', height=500)

## 5. Map 3 — Plotly Express choropleth (for comparison)

In [ ]:
import json

with open(GEOJSON) as f:
    geojson_dict = json.load(f)

imd_merged = imd.copy()

fig = px.choropleth(
    imd_merged,
    geojson=geojson_dict,
    locations='LAD22CD',
    featureidkey='properties.LAD22CD',
    color='imd_score',
    color_continuous_scale='YlOrRd',
    title='Index of Multiple Deprivation 2019 — Plotly Express',
    labels={'imd_score': 'IMD Score'},
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 6. Findings

- **IMD deprivation** is concentrated in northern English metropolitan areas (Blackpool, Middlesbrough, Kingston upon Hull) and inner London boroughs.
- **Unemployment claimant rates** broadly correlate with deprivation scores (r ≈ 0.7 across English LAs).
- The Plotly choropleth uses the same GeoJSON as Folium but renders inline in the notebook without an iframe — useful for quick EDA but produces larger notebook file sizes.
- Folium's `CartoDB positron` basemap provides a clean, low-contrast background that lets the choropleth colours read clearly.